# Data Cleaning and Preparation

## Library and Data Imports

In [ ]:
import pandas as pd
import numpy as np 
import ast
import string
from spellchecker import SpellChecker
import json

from nltk.stem import PorterStemmer
from IPython.display import display


stemmer = PorterStemmer()

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
clean_df = pd.read_excel('Data/Inbound/cleaned_terms_updated.xlsx', sheet_name = 'Cleaned Terms')
remove_df = pd.read_excel('Data/Inbound/cleaned_terms_updated.xlsx', sheet_name = 'Removed Terms')

survey_df = pd.read_csv('Data/Inbound/final_data_export.csv')


In [ ]:
print(f'Cleaned Terms: {clean_df.shape[0]} rows')
print(f'Removed Terms: {remove_df.shape[0]} rows')
print(f'Survey Data: {survey_df.shape[0]} rows, {survey_df.shape[1]} columns')

## Survey Data Preparation

In [ ]:
survey_df.head(3)

In [ ]:
question_df = survey_df[survey_df['StartDate'] == 'Start Date']
question_df.head(3)

In [ ]:
response_df = survey_df[(survey_df['StartDate']!= 'Start Date') & (survey_df['Status']!= '{"ImportId":"status"}')].reset_index(drop=True)
response_df.head(3)

In [ ]:
response_df.shape

## Date Filtering

In [ ]:
#Make date column

response_df['StartDateDTM'] = pd.to_datetime(response_df['StartDate'])
response_df['EndDateDTM'] = pd.to_datetime(response_df['EndDate'])  
response_df[['StartDateDTM', 'EndDateDTM']].head(3)

In [ ]:
response_df['StartDateYear'] = response_df['StartDateDTM'].dt.year
response_df['StartDateMonth'] = response_df['StartDateDTM'].dt.month
response_df['StartDateDay'] = response_df['StartDateDTM'].dt.day

response_df.groupby(by=['StartDateYear', 'StartDateMonth']).size()

In [ ]:
response_df = response_df[response_df['StartDateDTM']<= '2025-06-01'].reset_index(drop=True)
response_df.shape

## Completion Filtering

In [ ]:
response_df.groupby(by='Progress').size()

In [ ]:
response_df[response_df['Progress'] == '57'].head(3)

In [ ]:
response_df['Progress'] = pd.to_numeric(response_df['Progress'])
response_df = response_df[response_df['Progress'] >= 33].reset_index(drop=True)
response_df.shape

# Explode Survey Records

In [ ]:
response_df.columns.to_list()

### Recent Mood

In [ ]:
recent_q = ['Q2_1','Q2_2','Q2_3','Q2_4','Q2_5','Q2_6','Q2_7','Q2_8','Q2_9','Q2_10']
recent_categories = ['Q3_1','Q3_2','Q3_3','Q3_4','Q3_5','Q3_6','Q3_7','Q3_8','Q3_9','Q3_10']

In [ ]:
recent_q_terms = []
term_template = {
    'metadata.ResponseId': '',
    'metadata.Finished': '',
    'metadata.Progress': '',
    'metadata.TermTrack': '',
    'metadata.QuestionId': '',
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
    'QX_X.UserCategoryMapping': '',
    'QX.MultipleCategoryTerm': '',
    'QX#1_X.MultipleCategoryMapping': '',
    'QX#2_X_1.UserProvidedCategory': '',
    'QX#2_X_1.CleanedCategory': '',
}

In [ ]:
for idx, row in response_df.iterrows():
    # if the progress < 33, skip the row. They have not completed all of the modules for recent game mood terms
    if row['Progress'] < 33:
        continue
    try:
        for id,q in enumerate(recent_q):
            if pd.isna(row[q]):
                continue
            else:
                json_obj = term_template.copy()
                json_obj['metadata.TermTrack'] = 'Recent'
                json_obj['metadata.ResponseId'] = row['ResponseId']
                json_obj['metadata.Finished'] = row['Finished']
                json_obj['metadata.Progress'] = row['Progress']
                json_obj['metadata.QuestionId'] = q
                question_id = q.split('_')[1]

                multiple_categories = row['Q4']
                if multiple_categories != 'No':
                    multiple_category_terms = multiple_categories.split(',')
                    multiple_category_terms = [cat.strip('Mood Term') for cat in multiple_category_terms]   
                else: 
                    multiple_category_terms = []    

                #check if the mood term is NaN
                
                json_obj['QX_X.UserProvidedMoodTerm'] = row[q]

                #clean the mood term
                #step 1 - remove leading hashtag
                mood_term = row[q].lstrip('#').strip()
                #step 2 - lowercase
                mood_term = mood_term.lower()
                #step 3 - spell check
                spell = SpellChecker()
                mood_term_cleaned = spell.correction(mood_term)

                if mood_term_cleaned != None:
                    json_obj['QX_X.CleanedMoodTerm'] = mood_term_cleaned
                else: json_obj['QX_X.CleanedMoodTerm'] = mood_term
                    
                #gather the user mapped category
                json_obj['QX_X.UserCategoryMapping'] = row[recent_categories[id]]

                #identify multiple categories
                if question_id in multiple_category_terms:
                    json_obj['QX.MultipleCategoryTerm'] = True

                    if pd.notna(row[f'Q5#1_{int(question_id)+1}']):
                        #split the multiple categories by comma
                        json_obj['QX#1_X.MultipleCategoryMapping'] = row[f'Q5#1_{int(question_id)+1}'].split(',')
                    else: 
                        json_obj['QX#1_X.MultipleCategoryMapping'] = None

                    if pd.notna(row[f'Q5#2_{int(question_id)+1}_1']):
                        json_obj['QX#2_X_1.UserProvidedCategory'] = row[f'Q5#2_{int(question_id)+1}_1']
                        #clean the category
                        user_provided_category = row[f'Q5#2_{int(question_id)+1}_1'].strip().lower()
                        user_provided_category_cleaned = spell.correction(user_provided_category)

                        if mood_term_cleaned != None:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category_cleaned
                        else:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category
                    else:
                        json_obj['QX#2_X_1.UserProvidedCategory'] = None
                        json_obj['QX#2_X_1.CleanedCategory'] = None

                else:
                    json_obj['QX.MultipleCategoryTerm'] = False
                    json_obj['QX#1_X.MultipleCategoryMapping'] = None
                    json_obj['QX#2_X_1.UserProvidedCategory'] = None
                    json_obj['QX#2_X_1.CleanedCategory'] = None

                recent_q_terms.append(json_obj)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        break

In [ ]:
# convert the array to a DataFrame and save to CSV
recent_q_df = pd.DataFrame(recent_q_terms)
recent_q_df.to_csv('Data/Outbound/recent_q_terms.csv', index=False, header=True)

In [ ]:
recent_q_df.shape

In [ ]:
#test to ensure that all terms are captured
recent_terms_df = response_df[response_df['Progress'] >= 33][recent_q].copy()
total_recent_terms = pd.DataFrame(recent_terms_df[recent_q].count(axis=1))
total_recent_terms.rename(columns={0:'Total_Recent_Terms'}, inplace=True)
total_recent_terms.sum()

### Favorite Mood

In [ ]:
fav_q = ['Q6_1','Q6_2','Q6_3','Q6_4','Q6_5','Q6_6','Q6_7','Q6_8','Q6_9','Q6_10']
fav_categories = ['Q7_1','Q7_2','Q7_3','Q7_4','Q7_5','Q7_6','Q7_7','Q7_8','Q7_9','Q7_10']

In [ ]:
fav_q_terms = []
term_template = {
    'metadata.ResponseId': '',
    'metadata.Finished': '',
    'metadata.Progress': '',
    'metadata.TermTrack': '',
    'metadata.QuestionId': '',
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
    'QX_X.UserCategoryMapping': '',
    'QX.MultipleCategoryTerm': '',
    'QX#1_X.MultipleCategoryMapping': '',
    'QX#2_X_1.UserProvidedCategory': '',
    'QX#2_X_1.CleanedCategory': '',
}

In [ ]:
for idx, row in response_df.iterrows():
    # if the progress < 57, skip the row. They have not completed all of the modules for fav game mood terms
    if row['Progress'] < 57:
        continue
    try:
        for id,q in enumerate(fav_q):
            if pd.isna(row[q]):
                continue
            else:
                json_obj = term_template.copy()
                json_obj['metadata.TermTrack'] = 'Favorite'
                json_obj['metadata.ResponseId'] = row['ResponseId']
                json_obj['metadata.Finished'] = row['Finished']
                json_obj['metadata.Progress'] = row['Progress']
                json_obj['metadata.QuestionId'] = q
                question_id = q.split('_')[1]

                multiple_categories = row['Q8']
                if multiple_categories != 'No':
                    multiple_category_terms = multiple_categories.split(',')
                    multiple_category_terms = [cat.strip('Mood Term') for cat in multiple_category_terms]   
                else: 
                    multiple_category_terms = []    

                #check if the mood term is NaN
                
                json_obj['QX_X.UserProvidedMoodTerm'] = row[q]

                #clean the mood term
                #step 1 - remove leading hashtag
                mood_term = row[q].lstrip('#').strip()
                #step 2 - lowercase
                mood_term = mood_term.lower()
                #step 3 - spell check
                spell = SpellChecker()
                mood_term_cleaned = spell.correction(mood_term)

                if mood_term_cleaned != None:
                    json_obj['QX_X.CleanedMoodTerm'] = mood_term_cleaned
                else: json_obj['QX_X.CleanedMoodTerm'] = mood_term
                    
                #gather the user mapped category
                json_obj['QX_X.UserCategoryMapping'] = row[fav_categories[id]]

                #identify multiple categories
                if question_id in multiple_category_terms:
                    json_obj['QX.MultipleCategoryTerm'] = True

                    if pd.notna(row[f'Q9#1_{int(question_id)+1}']):
                        #split the multiple categories by comma
                        json_obj['QX#1_X.MultipleCategoryMapping'] = row[f'Q9#1_{int(question_id)+1}'].split(',')
                    else: 
                        json_obj['QX#1_X.MultipleCategoryMapping'] = None

                    if pd.notna(row[f'Q9#2_{int(question_id)+1}_1']):
                        json_obj['QX#2_X_1.UserProvidedCategory'] = row[f'Q9#2_{int(question_id)+1}_1']
                        #clean the category
                        user_provided_category = row[f'Q9#2_{int(question_id)+1}_1'].strip().lower()
                        user_provided_category_cleaned = spell.correction(user_provided_category)

                        if mood_term_cleaned != None:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category_cleaned
                        else:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category
                    else:
                        json_obj['QX#2_X_1.UserProvidedCategory'] = None
                        json_obj['QX#2_X_1.CleanedCategory'] = None

                else:
                    json_obj['QX.MultipleCategoryTerm'] = False
                    json_obj['QX#1_X.MultipleCategoryMapping'] = None
                    json_obj['QX#2_X_1.UserProvidedCategory'] = None
                    json_obj['QX#2_X_1.CleanedCategory'] = None

                fav_q_terms.append(json_obj)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        break

In [ ]:
# convert the array to a DataFrame and save to CSV
fav_q_df = pd.DataFrame(fav_q_terms)
fav_q_df.to_csv('Data/Outbound/fav_q_terms.csv', index=False, header=True)

In [ ]:
fav_q_df.shape

In [ ]:
#test to ensure that all terms are captured
fav_terms_df = response_df[response_df['Progress'] >= 57][fav_q].copy()
total_fav_terms = pd.DataFrame(fav_terms_df[fav_q].count(axis=1))
total_fav_terms.rename(columns={0:'Total_Favorite_Terms'}, inplace=True)
total_fav_terms.sum()

### Avoid Mood

In [ ]:
avoid_q = ['Q10_1','Q10_2','Q10_3','Q10_4','Q10_5','Q10_6','Q10_7','Q10_8','Q10_9','Q10_10']
avoid_categories = ['Q11_1','Q11_2','Q11_3','Q11_4','Q11_5','Q11_6','Q11_7','Q11_8','Q11_9','Q11_10']

In [ ]:
avoid_q_terms = []
term_template = {
    'metadata.ResponseId': '',
    'metadata.Finished': '',
    'metadata.Progress': '',
    'metadata.TermTrack': '',
    'metadata.QuestionId': '',
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
    'QX_X.UserCategoryMapping': '',
    'QX.MultipleCategoryTerm': '',
    'QX#1_X.MultipleCategoryMapping': '',
    'QX#2_X_1.UserProvidedCategory': '',
    'QX#2_X_1.CleanedCategory': '',
}

In [ ]:
for idx, row in response_df.iterrows():
    # if the progress < 100, skip the row. They have not completed all of the modules for fav game mood terms
    if row['Progress'] < 100:
        continue
    try:
        for id,q in enumerate(avoid_q):
            if pd.isna(row[q]):
                continue
            else:
                json_obj = term_template.copy()
                json_obj['metadata.TermTrack'] = 'Avoid'
                json_obj['metadata.ResponseId'] = row['ResponseId']
                json_obj['metadata.Finished'] = row['Finished']
                json_obj['metadata.Progress'] = row['Progress']
                json_obj['metadata.QuestionId'] = q
                question_id = q.split('_')[1]

                multiple_categories = row['Q12']
                if multiple_categories != 'No':
                    multiple_category_terms = multiple_categories.split(',')
                    multiple_category_terms = [cat.strip('Mood Term') for cat in multiple_category_terms]   
                else: 
                    multiple_category_terms = []    

                #check if the mood term is NaN
                
                json_obj['QX_X.UserProvidedMoodTerm'] = row[q]

                #clean the mood term
                #step 1 - remove leading hashtag
                mood_term = row[q].lstrip('#').strip()
                #step 2 - lowercase
                mood_term = mood_term.lower()
                #step 3 - spell check
                spell = SpellChecker()
                mood_term_cleaned = spell.correction(mood_term)

                if mood_term_cleaned != None:
                    json_obj['QX_X.CleanedMoodTerm'] = mood_term_cleaned
                else: json_obj['QX_X.CleanedMoodTerm'] = mood_term
                    
                #gather the user mapped category
                json_obj['QX_X.UserCategoryMapping'] = row[avoid_categories[id]]

                #identify multiple categories
                if question_id in multiple_category_terms:
                    json_obj['QX.MultipleCategoryTerm'] = True

                    if pd.notna(row[f'Q13#1_{int(question_id)+1}']):
                        #split the multiple categories by comma
                        json_obj['QX#1_X.MultipleCategoryMapping'] = row[f'Q13#1_{int(question_id)+1}'].split(',')
                    else: 
                        json_obj['QX#1_X.MultipleCategoryMapping'] = None

                    if pd.notna(row[f'Q13#2_{int(question_id)+1}_1']):
                        json_obj['QX#2_X_1.UserProvidedCategory'] = row[f'Q13#2_{int(question_id)+1}_1']
                        #clean the category
                        user_provided_category = row[f'Q13#2_{int(question_id)+1}_1'].strip().lower()
                        user_provided_category_cleaned = spell.correction(user_provided_category)

                        if mood_term_cleaned != None:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category_cleaned
                        else:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category
                    else:
                        json_obj['QX#2_X_1.UserProvidedCategory'] = None
                        json_obj['QX#2_X_1.CleanedCategory'] = None

                else:
                    json_obj['QX.MultipleCategoryTerm'] = False
                    json_obj['QX#1_X.MultipleCategoryMapping'] = None
                    json_obj['QX#2_X_1.UserProvidedCategory'] = None
                    json_obj['QX#2_X_1.CleanedCategory'] = None

                avoid_q_terms.append(json_obj)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        break

In [ ]:
# convert the array to a DataFrame and save to CSV
avoid_q_df = pd.DataFrame(avoid_q_terms)
avoid_q_df.to_csv('Data/Outbound/avoid_q_terms.csv', index=False, header=True)

In [ ]:
avoid_q_df.shape

In [ ]:
#test to ensure that all terms are captured
avoid_terms_df = response_df[response_df['Progress'] >= 100][avoid_q].copy()
total_avoid_terms = pd.DataFrame(avoid_terms_df[avoid_q].count(axis=1))
total_avoid_terms.rename(columns={0:'Total_Avoid_Terms'}, inplace=True)
total_avoid_terms.sum()

In [ ]:
term_df = pd.concat([recent_q_df, fav_q_df, avoid_q_df], ignore_index=True)
term_df.reset_index(drop=True, inplace=True)
term_df.to_csv('Data/Outbound/all_terms.csv', index=False, header=True)
term_df.to_excel('Data/Outbound/all_terms.xlsx', index=False, header=True)

In [ ]:
term_df.shape

## Stemming

In [ ]:
#duplicate the base dataframe to avoid modifying the original
temp_df = term_df.copy()
temp_df.insert(7, 'QX_X.CleanedTermStem', '', True)
temp_df.insert(8, 'QX_X.RemovedTermFlag', '', True)

# remove all whitespace from the term columns
temp_df['QX_X.UserProvidedMoodTerm'] = temp_df['QX_X.UserProvidedMoodTerm'].str.strip()
temp_df['QX_X.CleanedMoodTerm'] = temp_df['QX_X.CleanedMoodTerm'].str.strip()

In [ ]:
temp_df.head(3)

In [ ]:
#duplicate the base dataframe to avoid modifying the original
clean_temp_df = clean_df.copy()

# remove all whitespace from the term columns
clean_temp_df['QX_X.UserProvidedMoodTerm'] = clean_temp_df['QX_X.UserProvidedMoodTerm'].str.strip()
clean_temp_df['QX_X.CleanedMoodTerm'] = clean_temp_df['QX_X.CleanedMoodTerm'].str.strip()

In [ ]:
clean_temp_df.head(3)

In [ ]:
#duplicate the base dataframe to avoid modifying the original
remove_temp_df = remove_df.copy()

# remove all whitespace from the term columns
remove_temp_df['QX_X.UserProvidedMoodTerm'] = remove_temp_df['QX_X.UserProvidedMoodTerm'].str.strip()
remove_temp_df['QX_X.CleanedMoodTerm'] = remove_temp_df['QX_X.CleanedMoodTerm'].str.strip()

In [ ]:
remove_temp_df.head(3)

In [ ]:
for idx, row in temp_df.iterrows():
    user_provided_mood_term = row['QX_X.UserProvidedMoodTerm']
    cleaned_mood_term = row['QX_X.CleanedMoodTerm']

    #removed term check
    if user_provided_mood_term in remove_temp_df['QX_X.UserProvidedMoodTerm'].values or cleaned_mood_term in remove_temp_df['QX_X.CleanedMoodTerm'].values:

        #update values
        temp_df.at[idx, 'QX_X.RemovedTermFlag'] = True
        temp_df.at[idx, 'QX_X.CleanedMoodTerm'] = remove_temp_df.loc[
            remove_temp_df['QX_X.UserProvidedMoodTerm'] == user_provided_mood_term,
            'QX_X.CleanedMoodTerm'
        ].iloc[0]

        #stem the cleaned mood term
        stemmed_term = stemmer.stem(temp_df.at[idx, 'QX_X.CleanedMoodTerm'])
        temp_df.at[idx, 'QX_X.CleanedTermStem'] = stemmed_term

        #print results
        print(f"REMOVED Term Found: '{user_provided_mood_term}' mapped to Cleaned Term: '{temp_df.at[idx, 'QX_X.CleanedMoodTerm']}' with Stem: '{stemmed_term}'")
    elif user_provided_mood_term not in remove_temp_df['QX_X.UserProvidedMoodTerm'].values and cleaned_mood_term not in remove_temp_df['QX_X.CleanedMoodTerm'].values:

        #update values
        temp_df.at[idx, 'QX_X.RemovedTermFlag'] = False
        temp_df.at[idx, 'QX_X.CleanedMoodTerm'] = clean_temp_df.loc[
            clean_temp_df['QX_X.UserProvidedMoodTerm'] == user_provided_mood_term, 'QX_X.CleanedMoodTerm'].iloc[0]

        #stem the cleaned mood term
        stemmed_term = stemmer.stem(temp_df.at[idx, 'QX_X.CleanedMoodTerm'])
        temp_df.at[idx, 'QX_X.CleanedTermStem'] = stemmed_term

        #print results
        print(f"KEEP TERM: '{user_provided_mood_term}' with Cleaned Term: '{cleaned_mood_term}' and Stem: '{stemmed_term}'")    

In [ ]:
temp_df.head(10)

In [ ]:
temp_df.to_csv('Data/Outbound/all_terms_cleaned_stemmed.csv', index=False, header=True)
temp_df.to_excel('Data/Outbound/all_terms_cleaned_stemmed.xlsx', index=False, header=True)

## General Testing

In [ ]:
temp_df[temp_df['QX_X.RemovedTermFlag'] == True].shape

In [ ]:
temp_df[temp_df['QX_X.RemovedTermFlag'] == False].shape